# AgRob tomato DETR: GPU training and count evaluation
Run cells from top to bottom after reading `AGROB_FINETUNE_GUIDE.md` and reviewing the provisional labels. In Colab, select **Runtime → Change runtime type → GPU** first. Upload `agrob-finetune-gpu-bundle.zip` to the top level of **My Drive**. Results are saved in `MyDrive/agrob_finetune_runs/`. Each run needs a new name.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
drive.mount('/content/drive')
bundle = Path('/content/drive/MyDrive/agrob-finetune-gpu-bundle.zip')
work = Path('/content/agrob_experiment')
assert bundle.is_file(), f'Upload the bundle first: {bundle}'
assert not work.exists(), 'Restart the Colab runtime to unpack a clean bundle'
work.mkdir()
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(work)
manifest = json.loads((work / 'bundle_manifest.json').read_text())
for item in manifest['files']:
    actual = hashlib.sha256((work / item['path']).read_bytes()).hexdigest()
    assert actual == item['sha256'], item['path']
os.chdir(work)
print(f'Unpacked and verified {len(manifest["files"])} files in {work}')


The next cell installs the object-detection libraries while keeping Colab's GPU PyTorch installation. It then confirms that the runtime can use CUDA.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.50,<5', 'timm>=1,<2', 'scipy>=1.12,<2', 'Pillow>=10,<12'], check=True)
import torch, transformers
print('PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab, reconnect, then run from the first cell'
print('GPU:', torch.cuda.get_device_name(0))


Set `prepared_name` to `agrob_finetune_v2` if the team has reviewed labels and rebuilt the bundle with corrections. Choose a fresh `run_name` for every training attempt. The default 480-pixel, 15-epoch run trains on all 237 training images and validates on all 40 validation images. If GPU memory is exhausted, start a new run at 320 pixels.

In [ ]:
prepared_name = 'agrob_finetune_v1'
run_name = 'tomato_detr_v1_full'
image_size = 480
epochs = 15
prepared = work / 'data' / prepared_name
runs = Path('/content/drive/MyDrive/agrob_finetune_runs')
runs.mkdir(exist_ok=True)
run = runs / run_name
assert prepared.is_dir(), f'Missing prepared data: {prepared}'
assert not run.exists(), f'Choose a new run_name: {run}'
summary = json.loads((prepared / 'summary.json').read_text())
print('Label status:', summary['label_review_status'])
print('Training:', summary['splits']['train']['images'], 'Validation:', summary['splits']['valid']['images'])
subprocess.run([sys.executable, 'finetune_agrob_fruit.py', 'train', '--prepared', str(prepared), '--epochs', str(epochs), '--image-size', str(image_size), '--batch-size', '1', '--grad-accum', '2', '--output', str(run)], check=True)
print('Best checkpoint:', run / 'best_model')


Validation chooses a confidence threshold using count error. Read the printed values and inspect saved previews before sharing an accuracy claim.

In [ ]:
valid_output = runs / (run_name + '_valid')
assert not valid_output.exists(), f'Choose a new output name: {valid_output}'
subprocess.run([sys.executable, 'finetune_agrob_fruit.py', 'evaluate', '--prepared', str(prepared), '--checkpoint', str(run / 'best_model'), '--split', 'valid', '--output', str(valid_output)], check=True)
valid_report = json.loads((valid_output / 'summary.json').read_text())
threshold = valid_report['chosen_metrics']['threshold']
print('Frozen threshold:', threshold, 'Validation count MAE:', valid_report['chosen_metrics']['mean_absolute_count_error'])
print('Review:', valid_output / 'thresholds.csv', 'and', valid_output / 'previews')


The development set was previously inspected during baseline work. This is a comparison, not a fresh final test. This cell uses the frozen validation threshold.

In [ ]:
development_output = runs / (run_name + '_development')
assert not development_output.exists(), f'Choose a new output name: {development_output}'
subprocess.run([sys.executable, 'finetune_agrob_fruit.py', 'evaluate', '--prepared', str(prepared), '--checkpoint', str(run / 'best_model'), '--split', 'development', '--thresholds', str(threshold), '--output', str(development_output)], check=True)
development_report = json.loads((development_output / 'summary.json').read_text())
print('Development count MAE:', development_report['chosen_metrics']['mean_absolute_count_error'])
print('Original pretrained Tomato-class baseline count MAE: 12.118 fruit/image')
print('All artifacts:', runs)
